# Xarray-Spatial Proximity: Balanced allocation

Standard `allocation` assigns each raster cell to the nearest source by cost distance, but the resulting territories can be wildly uneven when friction varies across the landscape. `balanced_allocation` adjusts those boundaries iteratively so each source ends up responsible for a similar share of the total workload, measured by cost-weighted area.

### What you'll build

1. Create source points and a friction surface
2. Compare standard allocation against balanced allocation
3. See how asymmetric friction shifts territory boundaries
4. Run balanced allocation with three sources
5. Tune convergence with the `tolerance` parameter
6. Block paths with NaN barriers

![Balanced allocation preview](images/balanced_allocation_preview.png)

[Standard vs balanced](#Standard-vs-balanced-allocation) · [Asymmetric friction](#Asymmetric-friction) · [Three sources](#Three-sources) · [Tuning tolerance](#Tuning-tolerance) · [Barriers](#Barriers)

Standard imports plus `scipy.ndimage` for generating a smooth friction field.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from scipy.ndimage import gaussian_filter

import xrspatial
from xrspatial import allocation
from xrspatial.balanced_allocation import balanced_allocation

## Data

All sections below reuse the same helper function and the same smooth friction field. The friction surface has values between 1 and 5, blurred with a Gaussian kernel so the cost landscape looks natural rather than random.

In [ ]:
def make_raster(data, res=1.0):
    """Create a DataArray with y/x coordinates."""
    h, w = data.shape
    raster = xr.DataArray(
        data.astype(np.float64),
        dims=['y', 'x'],
        attrs={'res': (res, res)},
    )
    raster['y'] = np.arange(h) * res
    raster['x'] = np.arange(w) * res
    return raster

def territory_stats(alloc_arr, friction_arr):
    """Print cell count and cost-weighted area per territory."""
    alloc = alloc_arr.values
    fric = friction_arr.values
    ids = sorted(np.unique(alloc[np.isfinite(alloc)]))
    print(f"{'Source':>8}  {'Cells':>6}  {'Cost-weighted area':>18}")
    print('-' * 38)
    for sid in ids:
        mask = alloc == sid
        n = int(np.sum(mask))
        w = float(np.sum(fric[mask]))
        print(f"{sid:>8.0f}  {n:>6d}  {w:>18.1f}")

# Shared friction field used in later sections
np.random.seed(42)
GRID = 40
raw_fric = np.random.uniform(1.0, 5.0, (GRID, GRID))
smooth_fric = gaussian_filter(raw_fric, sigma=3)
smooth_fric = np.clip(smooth_fric, 1.0, None)
friction_base = make_raster(smooth_fric)

# Shared territory colormap (colorblind-safe Set2)
territory_cmap = ListedColormap(plt.cm.Set2(np.linspace(0, 1, 8))[:3])

The friction surface represents travel cost per cell. Higher values (brighter) are more expensive to cross.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
friction_base.plot.imshow(ax=ax, cmap='YlOrRd', add_colorbar=True,
                          cbar_kwargs={'label': 'Friction', 'shrink': 0.7})
ax.set_title('Smooth friction surface')
ax.set_axis_off()

## Standard vs balanced allocation

Standard `allocation` assigns each cell to the nearest source by cost distance. When sources aren't evenly spaced, one territory can end up much larger than the other. `balanced_allocation` iteratively nudges the boundaries until each source's cost-weighted area (sum of friction values within its territory) is roughly equal. See the [algorithm description](https://xarray-spatial.org/reference/_autosummary/xrspatial.balanced_allocation.balanced_allocation.html) in the API docs.

The two plots below show the same two sources on a uniform friction grid. Standard allocation gives the off-centre source a smaller zone; balanced allocation evens things out.

In [ ]:
# Two sources placed asymmetrically on a 30x30 grid
source_data = np.zeros((30, 30))
source_data[5, 5] = 1.0    # source near top-left corner
source_data[15, 15] = 2.0  # source near centre

raster = make_raster(source_data)
friction_uniform = make_raster(np.ones((30, 30)))
locs = [(5, 5), (15, 15)]
cmap2 = ListedColormap(plt.cm.Set2(np.linspace(0, 1, 8))[:2])

# Standard allocation (nearest by cost)
std_alloc = allocation(raster)

# Balanced allocation
bal_alloc = balanced_allocation(raster, friction_uniform, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(10, 7.5))
for ax, arr, label in [(axes[0], std_alloc, 'Standard allocation'),
                       (axes[1], bal_alloc, 'Balanced allocation')]:
    arr.plot.imshow(ax=ax, cmap=cmap2, add_colorbar=False)
    for r, c in locs:
        ax.plot(c, r, 'k*', markersize=12)
    ax.set_title(label)
    ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor=cmap2(0), label='Source 1'),
                       Patch(facecolor=cmap2(1), label='Source 2')],
              loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()

print('Standard allocation:')
territory_stats(std_alloc, friction_uniform)
print('\nBalanced allocation:')
territory_stats(bal_alloc, friction_uniform)

The balanced version shifts the boundary so both territories end up with a similar cell count. Since friction is uniform here, cell count and cost-weighted area are the same thing.

## Asymmetric friction

When friction varies across the grid, "balanced" no longer means equal cell counts. The algorithm balances *cost-weighted area*: the territory covering expensive cells gets fewer of them, while the territory covering cheap cells gets more. The total friction within each zone converges to the same value.

Below, the left half has friction 1 and the right half has friction 4. The plot shows the friction surface, the cost distance field, and the balanced territories side by side.

In [ ]:
# Left half: low friction. Right half: high friction.
source_data2 = np.zeros((30, 30))
source_data2[15, 5] = 1.0   # source on the cheap side
source_data2[15, 25] = 2.0  # source on the expensive side

fric_data2 = np.ones((30, 30))
fric_data2[:, 15:] = 4.0  # right half is 4x more costly

raster2 = make_raster(source_data2)
friction2 = make_raster(fric_data2)
locs2 = [(15, 5), (15, 25)]

bal2 = balanced_allocation(raster2, friction2, tolerance=0.05)
cd2 = xrspatial.cost_distance(raster2, friction2)

fig, axes = plt.subplots(1, 3, figsize=(10, 7.5))

# Friction surface
friction2.plot.imshow(ax=axes[0], cmap='YlOrRd', add_colorbar=True,
                      cbar_kwargs={'shrink': 0.5})
axes[0].set_title('Friction surface')
axes[0].set_axis_off()

# Cost distance
cd2.plot.imshow(ax=axes[1], cmap='magma', add_colorbar=True,
                cbar_kwargs={'shrink': 0.5})
axes[1].set_title('Cost distance')
axes[1].set_axis_off()

# Balanced territories
bal2.plot.imshow(ax=axes[2], cmap=cmap2, add_colorbar=False)
for r, c in locs2:
    axes[2].plot(c, r, 'k*', markersize=12)
axes[2].set_title('Balanced allocation')
axes[2].set_axis_off()
axes[2].legend(handles=[Patch(facecolor=cmap2(0), label='Source 1 (cheap side)'),
                        Patch(facecolor=cmap2(1), label='Source 2 (expensive side)')],
               loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()

print('Balanced allocation territory stats:')
territory_stats(bal2, friction2)

Source 1 on the cheap side covers more cells, but the total friction within each territory is roughly equal. That is the point of balancing by cost-weighted area rather than raw cell count.

<div class="alert alert-block alert-warning">
<b>Cost-weighted area depends on your friction units.</b> The balancing target is the sum of friction values within each territory. If your friction surface is in seconds-per-meter, the territories balance on total travel time. If it is unitless difficulty, they balance on total difficulty. Make sure the friction values represent the quantity you actually want to equalize.
</div>

## Three sources

The algorithm scales to any number of sources. Each territory converges to roughly 1/N of the total cost-weighted area. Here we use the smooth friction field from the data section and place three sources around the grid.

The plot shows the friction surface alongside the balanced territories.

In [ ]:
# Three sources on the shared friction grid
source_data3 = np.zeros((GRID, GRID))
source_data3[8, 20] = 1.0
source_data3[30, 8] = 2.0
source_data3[30, 32] = 3.0

raster3 = make_raster(source_data3)
locs3 = [(8, 20), (30, 8), (30, 32)]

bal3 = balanced_allocation(raster3, friction_base, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(10, 7.5))

# Friction surface with source markers
friction_base.plot.imshow(ax=axes[0], cmap='YlOrRd', add_colorbar=True,
                          cbar_kwargs={'label': 'Friction', 'shrink': 0.7})
for r, c in locs3:
    axes[0].plot(c, r, 'k*', markersize=12)
axes[0].set_title('Friction surface')
axes[0].set_axis_off()

# Balanced territories
bal3.plot.imshow(ax=axes[1], cmap=territory_cmap, add_colorbar=False)
for r, c in locs3:
    axes[1].plot(c, r, 'k*', markersize=12)
axes[1].set_title('Balanced allocation (3 sources)')
axes[1].set_axis_off()
axes[1].legend(handles=[Patch(facecolor=territory_cmap(0), label='Source 1'),
                        Patch(facecolor=territory_cmap(1), label='Source 2'),
                        Patch(facecolor=territory_cmap(2), label='Source 3')],
               loc='lower right', fontsize=11, framealpha=0.9)

plt.tight_layout()

print('Territory stats:')
territory_stats(bal3, friction_base)

## Tuning tolerance

The `tolerance` parameter sets how close to equal the territories need to be, measured as the maximum fractional deviation from the mean cost-weighted area. Lower tolerance produces tighter balance but takes more iterations. The `learning_rate` controls how aggressively biases shift each step: smaller values are more stable, larger values converge faster but can overshoot.

The three plots below show the same three-source problem at tolerance 0.20, 0.05, and 0.01.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 7.5))

for ax, tol in zip(axes, [0.20, 0.05, 0.01]):
    result = balanced_allocation(
        raster3, friction_base, tolerance=tol, max_iterations=200,
    )
    result.plot.imshow(ax=ax, cmap=territory_cmap, add_colorbar=False)
    for r, c in locs3:
        ax.plot(c, r, 'k*', markersize=12)
    ax.set_title(f'tolerance = {tol}')
    ax.set_axis_off()

    # Compute balance quality
    vals = result.values
    weights = []
    for sid in [1.0, 2.0, 3.0]:
        weights.append(float(np.sum(smooth_fric[vals == sid])))
    mean_w = np.mean(weights)
    max_dev = max(abs(w - mean_w) / mean_w for w in weights)
    ax.set_xlabel(f'max deviation: {max_dev:.1%}')

axes[0].legend(handles=[Patch(facecolor=territory_cmap(0), label='Source 1'),
                        Patch(facecolor=territory_cmap(1), label='Source 2'),
                        Patch(facecolor=territory_cmap(2), label='Source 3')],
               loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()

## Barriers

NaN cells in the friction surface act as impassable barriers, the same way they do in `cost_distance`. The balanced allocation routes around them and leaves unreachable cells as NaN in the output.

Below, a diagonal wall of NaN cells splits the grid, with a narrow gap near the centre. Both sources must route through that gap.

In [ ]:
# Grid with a diagonal barrier and a narrow gap
source_data4 = np.zeros((30, 30))
source_data4[5, 3] = 1.0
source_data4[25, 27] = 2.0

fric_data4 = np.ones((30, 30))
for i in range(30):
    if abs(i - 15) > 2:  # gap near the middle
        fric_data4[i, i] = np.nan
        if i + 1 < 30:
            fric_data4[i, i + 1] = np.nan

raster4 = make_raster(source_data4)
friction4 = make_raster(fric_data4)
locs4 = [(5, 3), (25, 27)]

bal4 = balanced_allocation(raster4, friction4, tolerance=0.05)

fig, axes = plt.subplots(1, 2, figsize=(10, 7.5))

# Friction with barrier visible
barrier_vis = make_raster(np.where(np.isnan(fric_data4), 0.0, fric_data4))
barrier_vis.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
for r, c in locs4:
    axes[0].plot(c, r, color='salmon', marker='*', markersize=12)
axes[0].set_title('Friction (dark = barrier)')
axes[0].set_axis_off()

# Balanced territories
bal4.plot.imshow(ax=axes[1], cmap=cmap2, add_colorbar=False)
for r, c in locs4:
    axes[1].plot(c, r, 'k*', markersize=12)
axes[1].set_title('Balanced allocation with barrier')
axes[1].set_axis_off()
axes[1].legend(handles=[Patch(facecolor=cmap2(0), label='Source 1'),
                        Patch(facecolor=cmap2(1), label='Source 2')],
               loc='lower right', fontsize=11, framealpha=0.9)

plt.tight_layout()

print('Territory stats:')
territory_stats(bal4, friction4)

Both sources route through the gap. The balanced allocation still equalizes cost-weighted area on each side, subject to the constraint that all paths must squeeze through that opening.

<div class="alert alert-block alert-info">
<b>Barriers vs high friction.</b> A NaN cell is truly impassable: cost distance cannot cross it at all. If you want a feature that is hard but not impossible to cross (a river, a steep ridge), use a high friction value instead of NaN. The distinction matters when balancing territories, because a barrier can strand cells on one side entirely.
</div>

In [ ]:
# Save preview image for the "What you'll build" cell

fig, ax = plt.subplots(figsize=(10, 7.5))
bal3.plot.imshow(ax=ax, cmap=territory_cmap, add_colorbar=False)
for r, c in locs3:
    ax.plot(c, r, 'k*', markersize=14)
ax.set_axis_off()
ax.legend(handles=[Patch(facecolor=territory_cmap(0), label='Source 1'),
                   Patch(facecolor=territory_cmap(1), label='Source 2'),
                   Patch(facecolor=territory_cmap(2), label='Source 3')],
          loc='lower right', fontsize=11, framealpha=0.9)
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/balanced_allocation_preview.png',
            bbox_inches='tight', dpi=120)
plt.close(fig)

### References

- [Cost distance analysis](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/understanding-cost-distance-analysis.htm), ArcGIS Pro documentation
- [Allocation analysis](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/how-cost-allocation-works.htm), ArcGIS Pro documentation
- [Service area partitioning for facility location](https://en.wikipedia.org/wiki/Facility_location_problem), Wikipedia